In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
    
bronze_table = "agriculture_dev.bronze.agriculture_raw"
silver_table = "agriculture_dev.silver.agriculture_clean"
quarantine_table = "agriculture_dev.silver.agriculture_quarantine"

bronze_df = spark.table(bronze_table)
# Process all batches; latest accepted observation per farm becomes current Silver state.

silver_df = bronze_df.select(
    trim("Farm_ID").alias("farm_id"),
    initcap(trim(("Crop_Type"))).alias("crop_type"),
    col("Farm_Area_acres").alias("farm_area_acres"),
    initcap(trim("Irrigation_Type")).alias("irrigation_type"),
    col("Fertilizer_Used_tons").alias("fertilizer_used_tons"),
    col("Pesticide_Used_kg").alias("pesticide_used_kg"),
    col("Yield_tons").alias("yield_tons"),
    initcap(trim("Soil_Type")).alias("soil_type"),
    initcap(trim("Season")).alias("season"),
    col("Water_Usage_cubic_meters").alias("water_usage_cubicmeters"),
    col("row_hash"),
    col("Src_name").alias("source_name"),
    col("Ingestion_timestamp").alias("ingestion_timestamp"),
    col("batch_id")
)

display(silver_df)
silver_df.printSchema()

validated_df = silver_df.withColumn(
    "validation_errors",
    concat_ws(
        ", ",
        when(col("farm_id").isNull() | (trim(col("farm_id")) == ""),"Missing Farm_ID"),
        when(col("crop_type").isNull() | (trim(col("crop_type")) == ""),"Missing Crop_Type"),
        when(col("farm_area_acres").isNull() | (col("farm_area_acres") < 0), "Invalid Farm_Area or less farm area"),
        when(col("fertilizer_used_tons").isNull() |(col("fertilizer_used_tons") < 0),"Invalid Fertilizer_Used"),
        when(col("pesticide_used_kg").isNull() |(col("pesticide_used_kg") < 0), "Invalid Pesticide_Used"),
        when(col("yield_tons").isNull() |(col("yield_tons") < 0),"Invalid Yield"),
        when(col("water_usage_cubicmeters").isNull() |(col("water_usage_cubicmeters") < 0),"Invalid Water_Usage"),
        when(col("irrigation_type").isNull() | (trim(col("irrigation_type")) == ""),"Missing Irrigation_Type"),
        when(col("soil_type").isNull() | (trim(col("soil_type")) == ""),"Missing Soil_Type"),
        when(col("season").isNull() |(trim(col("season")) == ""), "Missing Season" )))

display(validated_df)

valid_df = validated_df.filter(col("validation_errors") == "")

quarantine_df = validated_df.filter(col("validation_errors") != "")

print("Total records:", validated_df.count())
print("Valid records:", valid_df.count())
print("Quarantined records:", quarantine_df.count())


# Composite business key. Rule 3: same Farm_ID with a different crop/season/irrigation
# is a different business key and is naturally retained as a separate record.
dedup_key = ["farm_id", "crop_type", "season", "irrigation_type"]
measure_cols = ["farm_area_acres", "fertilizer_used_tons", "pesticide_used_kg", "yield_tons", "water_usage_cubicmeters"]

# NULL-safe signature of the measure columns so NULL and blank are compared explicitly
measure_sig = concat_ws("||", *[coalesce(col(c).cast("string"), lit("<NULL>")) for c in measure_cols])

key_stats = (valid_df.withColumn("measure_signature", measure_sig).groupBy(*dedup_key)
             .agg(count("*").alias("rows_in_group"),
                  count_distinct("measure_signature").alias("distinct_measures")))

# Rule 2 - same key, different measures: conflicting records, quarantine every member
conflict_keys = key_stats.filter(col("distinct_measures") > 1).select(*dedup_key)

# Rule 1 - same key, identical measures: exact duplicates, retain one per key
duplicate_keys = key_stats.filter((col("distinct_measures") == 1) & (col("rows_in_group") > 1)).select(*dedup_key)

# Rank within each key group: newest observation wins
w = Window.partitionBy(*dedup_key).orderBy(col("row_hash").asc(), col("batch_id").asc())
ranked = valid_df.withColumn("rn", row_number().over(w))

conflicting_records = (ranked.join(conflict_keys, on=dedup_key, how="inner")
                       .withColumn("validation_errors", lit("conflicting measures for the same business key")))

dup_group_records = ranked.join(duplicate_keys, on=dedup_key, how="inner")
retained_duplicates = dup_group_records.filter(col("rn") == 1).drop("rn")
dropped_duplicates = dup_group_records.filter(col("rn") > 1).drop("rn")

# Keys with no conflicts and no duplication pass straight through
unique_valid_df = (ranked.join(conflict_keys, on=dedup_key, how="left_anti")
                   .join(duplicate_keys, on=dedup_key, how="left_anti")
                   .drop("rn"))

# Rows that proceed to the Silver MERGE: unique keys + one retained row per exact-duplicate key
silver_ready_df = unique_valid_df.unionByName(retained_duplicates)

print("Valid records entering dedup:", valid_df.count())
print("Exact-duplicate groups (one retained each):", duplicate_keys.count())
print("Exact duplicate copies dropped:", dropped_duplicates.count())
print("Conflicting records quarantined:", conflicting_records.count())
print("Records proceeding to Silver MERGE:", silver_ready_df.count())


quarantine_df.write \
    .format("delta") \
    .mode("append") \
    .option("overwriteSchema", "true") \
    .saveAsTable(quarantine_table)

## apply merge
# Create the empty Silver target on the first run so the MERGE has a table to merge into
if not spark.catalog.tableExists(silver_table):
    silver_ready_df.write.format("delta").option("overwriteSchema", "true").saveAsTable(silver_table)

target = DeltaTable.forName(spark, silver_table)

merge_condition = """
    target.farm_id = source.farm_id
    AND target.crop_type = source.crop_type
    AND target.season = source.season
    AND target.irrigation_type = source.irrigation_type
"""

(
    target.alias("target")
    .merge(
        silver_ready_df.alias("source"),
        merge_condition
    )
    .whenMatchedUpdateAll(
        condition="target.row_hash <> source.row_hash"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver MERGE completed successfully")